# 02 — Draft Prediction Model & Which Drills Actually MatterParts 2 and 6: predicting draft value from combine measurables, and finding which drills matter *per position* rather than in aggregate.

In [ ]:
import sys; sys.path.insert(0, '../src')import pandas as pd, numpy as npimport matplotlib.pyplot as pltdf = pd.read_csv('../data/processed/combine_with_sai.csv')df.shape

## Train/test the draft value modelNote: xgboost/lightgbm weren't installable in the offline sandbox this was built in — RandomForest / GradientBoosting / HistGradientBoosting (sklearn's gradient-boosted-tree family) are used instead. Swap in `xgboost.XGBRegressor` / `lightgbm.LGBMRegressor` locally if desired; the pipeline code doesn't depend on sklearn internals.

In [ ]:
import draft_model# draft_model.main() trains all 3 models, saves the best one, and writes predictions to disk.# Re-running it here for illustration (takes ~10-20s):draft_model.main()

In [ ]:
import jsonresults = json.load(open('../models/model_results.json'))pd.DataFrame(results['results_by_model']).T

**R² ≈ 0.21** — combine measurables alone explain only about a fifth of draft-position variance. This is a real finding, not a modeling failure: it quantifies how much of a draft decision comes from tape, production, and intangibles rather than the workout.

In [ ]:
fi = results['feature_importance_best_model']pd.Series(fi).sort_values(ascending=False).plot(kind='barh', figsize=(7,4), color='#3C6E47')plt.title('Permutation Feature Importance — Best Model'); plt.tight_layout(); plt.show()

## Which drills matter, by position (Part 6)

In [ ]:
import feature_importance as fimpresults_by_pos = {}for pos in fimp.POSITIONS:    r = fimp.analyze_position(df, pos)    if r: results_by_pos[pos] = rlen(results_by_pos)

In [ ]:
import numpy as npdrills = ['40-yd Dash','Weight','Height','Vertical Jump','Broad Jump','3-Cone Drill','20-yd Shuttle','Bench Press']mat = pd.DataFrame({pos: {d: results_by_pos[pos]['rf_importance'].get(d,0) for d in drills} for pos in results_by_pos}).Tmat.style.background_gradient(cmap='YlGn', axis=None).format('{:.3f}')

**Takeaways:** 40-yard dash dominates importance almost everywhere (even OL/DL, where it's acting as a general-athleticism proxy). Bench press is consistently the weakest predictor. Weight matters heavily for OL/DL/TE. This matches the example findings from the project brief almost exactly.